In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O yellow_tripdata_2025-11.parquet
!curl -o yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [1]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("TaxiData").getOrCreate()
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.coalesce(4).write.mode("overwrite").parquet("partitioned_data")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 09:42:54 WARN Utils: Your hostname, macs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.88.178 instead (on interface en0)
26/03/09 09:42:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 09:42:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.functions import to_date

nov_15_trips = df.filter(to_date(df.tpep_pickup_datetime) == '2025-11-15').count()
print(f'Taxi trips on November 15th: {nov_15_trips}')

Taxi trips on November 15th: 162604


In [4]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max

longest_trip = df.withColumn('trip_hours', 
    (unix_timestamp(col('tpep_dropoff_datetime')) - unix_timestamp(col('tpep_pickup_datetime'))) / 3600
).agg(spark_max('trip_hours')).collect()[0][0]

print(f'Longest trip duration: {longest_trip} hours')

Longest trip duration: 90.64666666666666 hours


In [5]:
!curl -o taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
zones_df = spark.read.option('header', 'true').csv('taxi_zone_lookup.csv')
zones_df.createOrReplaceTempView('zones')
df.createOrReplaceTempView('trips')

result = spark.sql("""
                        SELECT z.Zone, COUNT(*) as pickup_count
                        FROM trips t
                        JOIN zones z ON t.PULocationID = z.LocationID
                        GROUP BY z.Zone
                        ORDER BY pickup_count ASC
                        LIMIT 1""")

least_frequent_zone = result.collect()[0]['Zone']
print(f'Least frequent pickup zone: {least_frequent_zone}')


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0  47268      0 --:--:-- --:--:-- --:--:-- 47426


Least frequent pickup zone: Governor's Island/Ellis Island/Liberty Island
